# Spark on the telemouse archive — slicing events into time ranges

`kafka2parquet.ipynb` turns the Kafka topics into a Hive-partitioned Parquet tree
under `data/`. This notebook picks up from there and uses **PySpark** to cut that
tree into ranges of time, five different ways:

| range type | question it answers | Spark tool |
|---|---|---|
| A. absolute window | "what happened between 21:40 and 21:50 UTC?" | integer filter on `ts_utc_us` (pushed into Parquet) |
| B. relative window | "the first 60 s of every session" | window function `min(...) over (partition by session_id)` |
| C. fixed buckets | "events per second / minute / 10 minutes" | `window(ts, "1 minute")`, tumbling and sliding |
| D. data-driven ranges | "bursts of activity", "how long was each game in the foreground", "which batches went missing" | `session_window`, `lag` + running sum |
| E. same thing in SQL | — | `spark.sql(...)` over a temp view |

Before any slicing, section 5 finds and repairs rows whose UTC timestamp the
archiver could not compute — a real defect in this archive, and the kind of thing
every time-range query silently gets wrong if nobody checks.

Every section is a markdown cell that explains the idea, a code cell that does
it, and a short *what to notice* list. Run the cells in order. The whole notebook
takes a couple of minutes on a laptop against ~3.5 M events.

**Honest framing.** At this data size DuckDB (section 6 of the other notebook) is
faster and simpler. Spark earns its keep when the archive no longer fits on one
machine, or when the same code has to run on a cluster later. Learning it on a
data set you understand — where you can check every answer against DuckDB — is
the point of this notebook.

## 1. Setup

Three things have to be in place before a `SparkSession` can start:

1. **Java 17 or 21.** Spark is a JVM program; PySpark drives it over a socket
   (py4j). `java -version` on the command line must work, or `JAVA_HOME` must
   point at a JDK.
2. **The `pyspark` package.** It bundles the Spark jars, so there is nothing else
   to download — about 400 MB.
3. **On Windows only: `winutils.exe` and `hadoop.dll`.** Spark reads local files
   through Hadoop's `RawLocalFileSystem`, and on Windows that class calls native
   code for permission checks. Without the DLL every `spark.read` fails with
   `UnsatisfiedLinkError: NativeIO$Windows.access0`. The community builds at
   [cdarlint/winutils](https://github.com/cdarlint/winutils) work with the Hadoop
   3.5 client that PySpark 4.x ships. The next cell downloads the two files into
   `hadoop/bin` next to this notebook (git-ignored) and points `HADOOP_HOME` at it.

The environment variables must be set **before** the JVM starts, i.e. before the
first `SparkSession.builder...getOrCreate()`. If you change them later, restart
the kernel.

In [ ]:
%pip install -q "pyspark>=4.0"

In [ ]:
import os
import sys
import time
import warnings
import urllib.request
import datetime as dt
from pathlib import Path

# PySpark 4.2 warns that pandas 3 is only partially supported. Everything this
# notebook does (toPandas on small results) works; silence the repeated warning.
warnings.filterwarnings("ignore", category=FutureWarning, module="pyspark")

HERE = Path.cwd()
assert (HERE / "kafka2parquet.ipynb").exists(), "run this notebook from tools/kafka2parquet"
DATA_DIR = HERE / "data"
ARCHIVE = DATA_DIR / "live" if (DATA_DIR / "live" / "events").exists() else DATA_DIR / "fixture"
OUT_DIR = DATA_DIR / "spark_out"

# --- Windows: Hadoop native shim ------------------------------------------
HADOOP_HOME = HERE / "hadoop"
if sys.platform == "win32":
    bin_dir = HADOOP_HOME / "bin"
    bin_dir.mkdir(parents=True, exist_ok=True)
    base = "https://raw.githubusercontent.com/cdarlint/winutils/master/hadoop-3.3.6/bin/"
    for name in ("winutils.exe", "hadoop.dll"):
        if not (bin_dir / name).exists():
            print("downloading", base + name)
            urllib.request.urlretrieve(base + name, bin_dir / name)
    os.environ["HADOOP_HOME"] = str(HADOOP_HOME)
    os.environ["PATH"] = str(bin_dir) + os.pathsep + os.environ["PATH"]   # so the JVM finds hadoop.dll

# Workers must run the same Python as this kernel (same venv, same packages).
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import pyspark
from pyspark.sql import SparkSession, Window, functions as F, types as T

print(sys.version.split()[0], "| pyspark", pyspark.__version__)
print("archive :", ARCHIVE)
print("JAVA_HOME:", os.environ.get("JAVA_HOME", "(not set — relying on java on PATH)"))

### The SparkSession

`SparkSession.builder` is where the cluster settings go. For a laptop:

| setting | value | why |
|---|---|---|
| `master` | `local[*]` | run driver and executors in this one JVM, one task per CPU core |
| `spark.sql.session.timeZone` | `UTC` | `ts_utc_us` is UTC. Setting the session zone to UTC makes every timestamp you see and every `window()` boundary UTC too. Otherwise Spark uses the machine zone and a "1 day" bucket starts at local midnight. |
| `spark.sql.shuffle.partitions` | `8` | the default is 200 — designed for clusters. Every `groupBy` on 200 partitions of a few MB each is pure overhead here. |
| `spark.sql.execution.arrow.pyspark.enabled` | `true` | `toPandas()` moves data through Arrow instead of pickling row by row |
| `spark.ui.showConsoleProgress` | `false` | progress bars in a notebook cell are noise; the web UI is better |

While the session is alive the **Spark UI** is at <http://localhost:4040>. It
shows every job, its stages and tasks, how many bytes each scan read, and the
physical plan. Keep it open in a tab while you work through the notebook.

In [ ]:
t0 = time.perf_counter()
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("telemouse-time-ranges")
    .config("spark.driver.host", "localhost")          # skip hostname lookups; UI at localhost:4040
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} up in {time.perf_counter() - t0:.1f} s")
print("UI:", spark.sparkContext.uiWebUrl)
print("cores:", spark.sparkContext.defaultParallelism)

## 2. Reading the Parquet tree

The archive is laid out as `events/date=YYYY-MM-DD/session_id=.../part-*.parquet`.
Point Spark at the top of the tree and it does **Hive partition discovery**: the
`date=` and `session_id=` path segments become columns, and the values never have
to be read from the files.

`spark.read.parquet` only opens the file footers to learn the schema. No row is
read until an *action* (`count`, `show`, `collect`, `write`) forces it. Everything
in between (`filter`, `select`, `groupBy`, `join`) just builds a plan.

**Unsigned integers.** Arrow wrote `seq_no` and `ts_qpc` as `uint64`. Parquet's
own type system has unsigned ints, but Spark's does not, so it widens each one to
the next signed type that can hold every value:

| Parquet (from pyarrow) | Spark |
|---|---|
| `uint8` (`device_ix`) | `short` |
| `uint16` (`buttons`) | `int` |
| `uint32` (`screen_w`, `drops_since_last`) | `long` |
| `uint64` (`seq_no`, `ts_qpc`) | `decimal(20,0)` |

Decimal arithmetic is slow and awkward. `seq_no` never gets near 2⁶³, so section 5
casts it to `long`. `ts_utc_us` was written as `int64` on purpose and arrives as
`long` already.

In [ ]:
events_raw   = spark.read.parquet((ARCHIVE / "events").as_posix())
sessions_raw = spark.read.parquet((ARCHIVE / "sessions").as_posix())
markers_raw  = spark.read.parquet((ARCHIVE / "markers").as_posix()) if (ARCHIVE / "markers").exists() else None

print("events:")
events_raw.printSchema()
print("sessions columns:", sessions_raw.columns)

In [ ]:
# The first action: one job, one task per file. Watch it appear in the Spark UI.
t0 = time.perf_counter()
n = events_raw.count()
print(f"{n:,} events in {time.perf_counter() - t0:.1f} s")

(events_raw
 .groupBy("date", "session_id")
 .agg(F.count("*").alias("events"),
      F.countDistinct("seq_no").alias("batches"),
      F.min("ts_utc_us").alias("first_us"),
      F.max("ts_utc_us").alias("last_us"))
 .withColumn("duration_s", F.round((F.col("last_us") - F.col("first_us")) / 1e6, 1))
 .orderBy("first_us")
 .show(truncate=False))

*What to notice*

- `printSchema` returned instantly; `count()` took a second or two and made a job
  with one task per Parquet file. Open the UI's **Jobs** tab and click through to
  the stage: the *Input* column shows how many bytes were actually read.
- `date` and `session_id` are at the end of the schema with the type Spark inferred
  from the path (`date` became a real `DateType`).
- The `groupBy` job has two stages: a scan stage and, after a **shuffle**, an
  aggregation stage with 8 tasks — the `spark.sql.shuffle.partitions` we set.

## 3. Lazy plans, partition pruning and predicate pushdown

Because nothing runs until an action, Spark can look at the *whole* plan first and
decide what not to read. Two optimisations matter for time ranges:

- **Partition pruning.** A filter on a partition column (`date`, `session_id`)
  removes whole directories from the file list before any file is opened.
- **Predicate pushdown.** A filter on a data column such as `ts_utc_us` is handed
  to the Parquet reader, which compares it against the min/max statistics stored
  per row group and skips row groups that cannot match. The filter still runs on
  the rows that do get read (that is the `Filter` node above the scan).

`explain()` prints the physical plan. Look for `PartitionFilters:` and
`PushedFilters:` in the `FileScan` line.

In [ ]:
some_date = events_raw.select(F.min("date")).first()[0]
print("partition filter on date =", some_date)
events_raw.filter(F.col("date") == some_date).explain()

cutoff_us = events_raw.select(F.max("ts_utc_us")).first()[0] - 60_000_000     # last 60 s of the archive
print("\ndata filter on ts_utc_us >", cutoff_us)
events_raw.filter(F.col("ts_utc_us") > cutoff_us).explain()

In [ ]:
# Proof that pruning happened: how many files did each plan touch?
def files_touched(df):
    return df.select(F.input_file_name()).distinct().count()

print("all files         :", files_touched(events_raw))
print("date == first day :", files_touched(events_raw.filter(F.col("date") == some_date)))
print("last 60 s         :", files_touched(events_raw.filter(F.col("ts_utc_us") > cutoff_us)))

*What to notice*

- The date filter shows up as `PartitionFilters: [isnotnull(date), (date = ...)]`
  and `PushedFilters: []`. The `Filter` node is gone entirely — Spark never has to
  evaluate it per row.
- The timestamp filter shows `PushedFilters: [IsNotNull(ts_utc_us), GreaterThan(ts_utc_us, ...)]`
  *and* a `Filter` node. The reader skips row groups by their statistics, then the
  filter rechecks each surviving row.
- The last count is smaller than the first even though no partition was named:
  files whose whole `ts_utc_us` range is below the cutoff produced no rows, so
  `input_file_name()` never saw them.

Rule of thumb for this archive: filter on `date` and/or `session_id` first when
you can, then on `ts_utc_us` as an **integer**. Converting to a timestamp and
filtering on that also works, but the pushdown is on the raw column.

## 4. Session metadata and a broadcast join

The `sessions` table is one row per session: when it started, the mouse CPI, the
coalesce window, and the **QPC anchor** — the pair `(anchor_qpc, anchor_utc_us)`
plus `qpc_freq` that `kafka2parquet.ipynb` section 3 used to turn every event's
QPC tick count into `ts_utc_us`. Section 5 needs that anchor, and later sections
need `mouse_cpi` (counts → centimetres), so join it onto the events now.

A join in Spark normally means shuffling both sides so matching keys land on the
same task. When one side is tiny — five rows here — it is far cheaper to send a
copy of it to every task and join locally. That is a **broadcast join**. Spark
does this automatically below a size threshold (10 MB by default); `F.broadcast`
makes the intent explicit.

In [ ]:
sessions = (sessions_raw
    .select("session_id", "started_utc_us", "mouse_cpi", "coalesce_ms",
            F.col("qpc_freq").cast("long").alias("qpc_freq"),
            F.col("anchor_qpc").cast("long").alias("anchor_qpc"),
            "anchor_utc_us")
    .withColumn("started", F.timestamp_micros("started_utc_us")))
sessions.orderBy("started").show(truncate=False)

evj = events_raw.join(F.broadcast(sessions), "session_id")

# path length in cm: counts / CPI * 2.54; kept as a column expression to reuse later
path_cm = F.sqrt(F.col("dx") ** 2 + F.col("dy") ** 2) / F.col("mouse_cpi") * 2.54

evj.explain()     # look for BroadcastHashJoin and BroadcastExchange

## 5. From microseconds to timestamps

`ts_utc_us` is microseconds since the Unix epoch, UTC. Spark's `TimestampType`
is also microsecond precision, so `timestamp_micros(ts_utc_us)` is an exact
conversion — no rounding.

### 5a. Rows with no timestamp

First a check that any real archive needs. The archiver computes `ts_utc_us` from
the session anchor, and the anchor arrives on a *different Kafka topic* than the
events. Batches consumed before their session envelope was seen get
`ts_utc_us = NULL` (the other notebook's section 10 counts them as
`rows_without_utc`). Spark's time functions do not fail on nulls — `window()`
quietly drops the rows — so an unnoticed null column means silently wrong
answers.

In [ ]:
(evj.groupBy("session_id")
    .agg(F.count("*").alias("events"),
         F.sum(F.col("ts_utc_us").isNull().cast("int")).alias("null_ts"))
    .withColumn("pct_null", F.round(F.col("null_ts") / F.col("events") * 100, 1))
    .orderBy("session_id")
    .show(truncate=False))

### 5b. Repairing them, bit-exactly

The fix is to redo what the archiver would have done had it known the anchor:

    ts_utc_us = anchor_utc_us + (ts_qpc − anchor_qpc) × 1 000 000 ÷ qpc_freq

with integer division. The archiver truncates toward zero; Spark's `div` on longs
also truncates, and the tick delta is never negative after the session starts, so
the two agree. The check below recomputes the column for *every* row and counts
how many differ from the stored value — expect zero. That is the standard way to
validate a repair: apply it where you already know the answer.

`tidy()` is the one place every conversion lives. Every later section starts from
`evs`, its output. It also casts the two `decimal(20,0)` columns to `long` and
adds a local wall-clock column; set `LOCAL_TZ` to your IANA zone.

In [ ]:
LOCAL_TZ = "America/New_York"      # change to your zone

def tidy(df):
    recomputed = (F.col("anchor_utc_us")
                  + F.expr("(ts_qpc - anchor_qpc) * 1000000L div qpc_freq"))
    return (df
        .withColumn("seq_no", F.col("seq_no").cast("long"))
        .withColumn("ts_qpc", F.col("ts_qpc").cast("long"))
        .withColumn("ts_utc_recomputed", recomputed)
        .withColumn("ts_was_null", F.col("ts_utc_us").isNull())
        .withColumn("ts_utc_us", F.coalesce("ts_utc_us", "ts_utc_recomputed"))
        .withColumn("ts", F.timestamp_micros("ts_utc_us"))
        .withColumn("ts_local", F.from_utc_timestamp("ts", LOCAL_TZ)))

evs = tidy(evj)

# Validation: where the archive already had a value, does the recomputation match it exactly?
(evs.filter(~F.col("ts_was_null"))
    .agg(F.count("*").alias("rows_checked"),
         F.sum((F.col("ts_utc_us") != F.col("ts_utc_recomputed")).cast("int")).alias("mismatches"),
         F.max(F.abs(F.col("ts_utc_us") - F.col("ts_utc_recomputed"))).alias("max_error_us"))
    .show())

print("null timestamps after repair:", evs.filter(F.col("ts").isNull()).count())
evs.select("session_id", "seq_no", "ts_utc_us", "ts", "ts_local", "dx", "dy", "game").show(5, truncate=False)

### 5c. UTC versus local

Two time zones show up in this data:

- **UTC** — what `ts_utc_us` means, what the session zone is set to, what the
  `date=` partition uses. A session that started at 21:36 local time on the 3rd
  lives in `date=2026-09-04` because that is its UTC date.
- **Local** — what you remember doing. `from_utc_timestamp(ts, LOCAL_TZ)` shifts
  the wall clock.

The cell makes the trap visible: sessions whose partition date differs from the
local date they started on.

In [ ]:
(evs.groupBy("date", "session_id")
    .agg(F.min("ts").alias("first_utc"), F.min("ts_local").alias("first_local"))
    .withColumn("local_date", F.to_date("first_local"))
    .withColumn("same_day", F.col("date") == F.col("local_date"))
    .orderBy("first_utc")
    .show(truncate=False))

*What to notice*

- The repair used only columns already on every row after the broadcast join;
  no shuffle, no window function. Look at `evs.explain()`: it is a single
  `Project` over the scan.
- `withColumn` with an existing name *replaces* the column, so `tidy()` records
  `ts_was_null` before overwriting `ts_utc_us`. Once overwritten, the original
  value is gone from that DataFrame.
- The 2026-09-05 session's first four minutes exist again. Every count from
  here on includes them; DuckDB queries against the raw files (section 12) do not.

## 6. Range type A — an absolute window

"Everything between two wall-clock instants." The cleanest way to express it is
to convert the two Python datetimes to microseconds and filter the integer
column, because that is the filter Parquet can push down (section 3).

Two details:

- Python `datetime` objects must be **aware** (have a `tzinfo`), otherwise
  `.timestamp()` silently assumes the machine zone.
- Use a half-open interval `[t0, t1)` so that adjacent windows never share an
  event.

In [ ]:
def to_us(t: dt.datetime) -> int:
    assert t.tzinfo is not None, "use an aware datetime"
    return int(t.timestamp() * 1_000_000)

def slice_abs(df, t0: dt.datetime, t1: dt.datetime):
    "Rows with t0 <= ts < t1. Filters the integer column so Parquet can skip row groups."
    return df.filter((F.col("ts_utc_us") >= to_us(t0)) & (F.col("ts_utc_us") < to_us(t1)))

# Pick the longest session and take ten minutes starting five minutes in.
longest = (evs.groupBy("session_id")
             .agg((F.max("ts_utc_us") - F.min("ts_utc_us")).alias("dur_us"), F.min("ts_utc_us").alias("t0_us"))
             .orderBy(F.desc("dur_us")).first())
start = dt.datetime.fromtimestamp(longest["t0_us"] / 1e6, tz=dt.timezone.utc)
t0, t1 = start + dt.timedelta(minutes=5), start + dt.timedelta(minutes=15)
print("session", longest["session_id"], "\nwindow ", t0, "→", t1)

win = slice_abs(evs, t0, t1)
(win.agg(F.count("*").alias("events"),
         F.min("ts").alias("first"), F.max("ts").alias("last"),
         F.round(F.sum(path_cm) / 100, 2).alias("path_m"),
         F.countDistinct("game").alias("games"))
    .show(truncate=False))
win.groupBy("game").count().orderBy(F.desc("count")).show(5)

*What to notice*

- Run `win.explain()` — the two comparisons appear under `PushedFilters`.
- Filtering on `ts` (the timestamp column we derived) instead would give the same
  rows, but the `FileScan` line would show `PushedFilters: []`, because `ts` is
  computed after the scan. On 3 M rows you will not feel the difference; on 3 B
  you would.

## 7. Range type B — relative to the start of each session

"The first 60 seconds of every session" is a filter whose threshold is different
per group. Two ways to get the per-session start:

1. Join the `sessions` table and use `started_utc_us` (we already did the join).
2. Compute it from the events with a **window function**:
   `min("ts_utc_us").over(Window.partitionBy("session_id"))`. This attaches the
   group minimum to every row *without* collapsing the rows, which a `groupBy`
   would do.

Option 2 works on any DataFrame, no metadata needed, so it is the one shown. The
cost is a shuffle by `session_id` — check the UI, there is an `Exchange` in the
plan.

In [ ]:
per_session = Window.partitionBy("session_id")

evr = (evs
    .withColumn("session_t0_us", F.min("ts_utc_us").over(per_session))
    .withColumn("t_rel_s", (F.col("ts_utc_us") - F.col("session_t0_us")) / 1e6))

def slice_rel(df, from_s: float, to_s: float):
    "Rows with from_s <= seconds-since-session-start < to_s, for every session."
    return df.filter((F.col("t_rel_s") >= from_s) & (F.col("t_rel_s") < to_s))

first_minute = slice_rel(evr, 0, 60)
(first_minute.groupBy("session_id")
    .agg(F.count("*").alias("events"),
         F.round(F.min("t_rel_s"), 3).alias("from_s"),
         F.round(F.max("t_rel_s"), 3).alias("to_s"),
         F.first("game").alias("first_game"))
    .orderBy("session_id")
    .show(truncate=False))

## 8. Range type C — fixed buckets (resampling)

Per-second, per-minute, per-10-minute rollups. Spark's `window(ts, "1 minute")`
assigns each row to a bucket and returns a struct with `start` and `end`. Buckets
are aligned to the Unix epoch in the *session time zone*, which is why section 1
set it to UTC — otherwise `"1 day"` buckets would start at local midnight and the
answer would change with the machine's zone.

Two flavours:

- **Tumbling** — `window(ts, "1 minute")`. Buckets do not overlap; every row is in
  exactly one.
- **Sliding** — `window(ts, "1 minute", "15 seconds")`. A new one-minute bucket
  starts every 15 s, so each row lands in four of them. Good for smoothed rates.

For plain integer bucketing, `floor(ts_utc_us / 1e6)` is the same thing as the
DuckDB query in `kafka2parquet.ipynb` section 6 and is slightly cheaper than
`window()`; the struct form is nicer to read and plot.

In [ ]:
def resample(df, every: str, slide: str | None = None):
    "Per-bucket events, path length, worst ring-buffer drop, per session."
    w = F.window("ts", every, slide) if slide else F.window("ts", every)
    return (df.groupBy("session_id", w.alias("w"))
              .agg(F.count("*").alias("events"),
                   F.round(F.sum(path_cm), 1).alias("path_cm"),
                   F.max("drops_since_last").alias("max_drops"),
                   F.countDistinct("game").alias("games"))
              .select("session_id", F.col("w.start").alias("start"), F.col("w.end").alias("end"),
                      "events", "path_cm", "max_drops", "games")
              .orderBy("session_id", "start"))

per_minute = resample(evs, "1 minute")
per_minute.filter(F.col("session_id") == longest["session_id"]).show(10, truncate=False)

for every in ("1 second", "10 seconds", "1 minute", "10 minutes"):
    print(f"{every:>11}: {resample(evs, every).count():>8,} buckets")

In [ ]:
# Sliding one-minute buckets every 15 s, for the ten-minute window from section 6.
(resample(win, "1 minute", "15 seconds")
    .select("start", "end", "events", "path_cm")
    .show(8, truncate=False))

# Same tumbling rollup with integer arithmetic — no struct, and identical numbers.
(win.groupBy((F.col("ts_utc_us") / 60_000_000).cast("long").alias("minute_no"))
    .agg(F.count("*").alias("events"))
    .withColumn("start", F.timestamp_seconds(F.col("minute_no") * 60))
    .orderBy("minute_no")
    .show(3))

*What to notice*

- `per_minute` is lazy too: nothing ran until `.show()` and each `.count()` in the
  loop re-ran the aggregation from the Parquet files. Section 12 shows `cache()`
  for when you reuse a DataFrame.
- The sliding version has about four times as many rows as the tumbling one, and
  each row's `events` is roughly four consecutive 15-s counts added together.

## 9. Range type D — ranges defined by the data itself

Not every interesting range has a fixed width. Three that come straight out of
the event stream:

**9a. Bursts.** Runs of mouse activity separated by idle gaps. Spark has this
built in as `session_window(ts, "5 seconds")`: a bucket stays open as long as the
next event arrives within 5 s of the previous one. The name is Spark's, nothing
to do with telemouse sessions.

**9b. Foreground game runs.** Consecutive events with the same `game` form a run.
This needs the manual pattern that underlies every "runs" query:

1. `lag(game)` — look at the previous row in time order (a window function with
   `partitionBy(session_id).orderBy(ts_utc_us)`).
2. `run_start = game != lag(game)` — mark the rows where a new run begins.
3. `run_id = sum(run_start) over (partition ... rows unbounded preceding)` — a
   running count of starts, which is the run number.
4. `groupBy(run_id)` for start, end, duration.

**9c. Missing batches.** `seq_no` is a per-session batch counter, so a jump of
more than one between consecutive distinct values is a range of batches that
never reached Kafka. Same `lag` trick on the distinct `seq_no` values, and the
`ts` of the batches either side bounds the lost time.

In [ ]:
# 9a — bursts via session_window
bursts = (evs.groupBy("session_id", F.session_window("ts", "5 seconds").alias("w"))
             .agg(F.count("*").alias("events"), F.round(F.sum(path_cm), 1).alias("path_cm"))
             .select("session_id", F.col("w.start").alias("start"), F.col("w.end").alias("end"),
                     "events", "path_cm")
             .withColumn("duration_s", F.round((F.col("end").cast("double") - F.col("start").cast("double")), 1)))

print("bursts per session:")
bursts.groupBy("session_id").agg(F.count("*").alias("bursts"),
                                 F.round(F.avg("duration_s"), 1).alias("avg_s"),
                                 F.max("duration_s").alias("longest_s")).orderBy("session_id").show(truncate=False)
print("longest bursts:")
bursts.orderBy(F.desc("duration_s")).show(5, truncate=False)

In [ ]:
# 9b — foreground-game runs with lag + running sum
by_time = Window.partitionBy("session_id").orderBy("ts_utc_us")
running = by_time.rowsBetween(Window.unboundedPreceding, Window.currentRow)

runs = (evs
    .withColumn("prev_game", F.lag("game").over(by_time))
    # eqNullSafe: the first row (prev_game is null) and rows with a null game still compare sensibly
    .withColumn("run_start", F.when(F.col("game").eqNullSafe(F.col("prev_game")), 0).otherwise(1))
    .withColumn("run_id", F.sum("run_start").over(running))
    .groupBy("session_id", "run_id", "game")
    .agg(F.min("ts").alias("start"), F.max("ts").alias("end"), F.count("*").alias("events"))
    .withColumn("duration_s", F.round(F.col("end").cast("double") - F.col("start").cast("double"), 1)))

print("time in foreground, summed over runs:")
(runs.groupBy("game")
     .agg(F.count("*").alias("runs"), F.round(F.sum("duration_s") / 60, 1).alias("minutes"), F.sum("events").alias("events"))
     .orderBy(F.desc("minutes")).show(10, truncate=False))
print("longest single runs:")
runs.orderBy(F.desc("duration_s")).select("session_id", "game", "start", "end", "duration_s", "events").show(5, truncate=False)

In [ ]:
# 9c — ranges of missing batches
by_seq = Window.partitionBy("session_id").orderBy("seq_no")

batches = (evs.groupBy("session_id", "seq_no")
             .agg(F.min("ts").alias("ts"))
             .withColumn("prev_seq", F.lag("seq_no").over(by_seq))
             .withColumn("prev_ts", F.lag("ts").over(by_seq)))

gaps = (batches.filter(F.col("seq_no") - F.col("prev_seq") > 1)
        .select("session_id",
                (F.col("prev_seq") + 1).alias("missing_from"), (F.col("seq_no") - 1).alias("missing_to"),
                (F.col("seq_no") - F.col("prev_seq") - 1).alias("missing_batches"),
                F.col("prev_ts").alias("last_seen"), F.col("ts").alias("next_seen"))
        .withColumn("hole_s", F.round(F.col("next_seen").cast("double") - F.col("last_seen").cast("double"), 3)))

n_gaps = gaps.count()
print(f"{n_gaps} gap(s) in seq_no across the archive")
if n_gaps:
    gaps.orderBy("session_id", "missing_from").show(20, truncate=False)

*What to notice*

- 9a needs no `orderBy`: `session_window` sorts within the aggregation. 9b and 9c
  do — a `lag` without an ordered window is an error, and the ordering *is* the
  definition of "previous".
- The `rowsBetween(unboundedPreceding, currentRow)` frame is what turns `sum` into
  a running total. Without the frame Spark defaults to a *range* frame, which
  gives every row with the same order value the same sum — wrong when two events
  share a `ts_utc_us`.
- Every window function here shuffles by `session_id`. Five sessions means five
  busy tasks and three idle ones; on a real cluster you would want a finer
  partition key.
- If 9c prints zero gaps it means section 12 of `kafka2parquet.ipynb` has already
  reconciled the archive from the JSONL recordings.

## 10. Range type E — the same thing in Spark SQL

Every DataFrame call above has a SQL spelling. Register a temp view and write the
per-minute rollup as a query. It compiles to the *same* physical plan — SQL and
the DataFrame API are two front ends for one optimiser — so pick whichever reads
better for the task. SQL is often clearer for range logic and for handing to
someone who knows DuckDB or Postgres.

In [ ]:
evs.createOrReplaceTempView("events")
sessions.createOrReplaceTempView("sessions")

sql_per_minute = spark.sql(f"""
    SELECT session_id,
           window(ts, '1 minute').start                                   AS start,
           count(*)                                                       AS events,
           round(sum(sqrt(dx*dx + dy*dy)) / mouse_cpi * 2.54, 1)          AS path_cm,
           max(drops_since_last)                                          AS max_drops
    FROM events
    WHERE session_id = '{longest["session_id"]}'
      AND ts_utc_us >= {to_us(t0)} AND ts_utc_us < {to_us(t1)}
    GROUP BY session_id, mouse_cpi, window(ts, '1 minute')
    ORDER BY start
""")
sql_per_minute.show(truncate=False)

# Cross-check against the DataFrame version from section 8.
df_per_minute = resample(win, "1 minute").select("session_id", "start", "events", "path_cm", "max_drops")
print("rows that differ:", df_per_minute.exceptAll(sql_per_minute).count())

## 11. Writing ranges back out

A rollup you will query again belongs on disk, not recomputed from 3 M rows each
time. `df.write.parquet` writes a *directory*, not a file: one `part-*.parquet`
per task, plus `_SUCCESS` and (on the local filesystem) `.crc` checksums.

- `partitionBy("date")` produces the same Hive layout the archive uses.
- `coalesce(1)` collapses to one file per partition; fine for small results,
  wrong for large ones because it also collapses the parallelism.
- `mode("overwrite")` replaces the whole output directory.

DuckDB reads the result straight back, which is the interoperability check that
matters: anything downstream that already reads `data/live` can read this too.

When you really want a single `.parquet` *file* — for a plot, for sharing — pull a
small result to the driver with `toArrow()` and let pyarrow write it.

In [ ]:
import shutil
import duckdb
import pyarrow.parquet as pq

out = OUT_DIR / "per_minute"
shutil.rmtree(out, ignore_errors=True)

t0_ = time.perf_counter()
(per_minute
    .withColumn("date", F.to_date("start"))
    .coalesce(1)
    .write.mode("overwrite").partitionBy("date").parquet(out.as_posix()))
print(f"wrote {out.relative_to(HERE)} in {time.perf_counter() - t0_:.1f} s")
for p in sorted(out.rglob("*.parquet")):
    print(f"  {p.relative_to(out).as_posix():<60} {p.stat().st_size/1024:6.1f} KB")

print("\nDuckDB reads it back:")
duckdb.sql(f"""
    SELECT date, session_id, count(*) AS minutes, sum(events) AS events, round(sum(path_cm)/100, 1) AS path_m
    FROM read_parquet('{(out / "**" / "*.parquet").as_posix()}', hive_partitioning=true)
    GROUP BY 1, 2 ORDER BY 1, 2
""").show()

# Single-file variant for a small result: driver-side Arrow table → pyarrow.
single = OUT_DIR / "bursts.parquet"
pq.write_table(bursts.toArrow(), single, compression="zstd")
print(f"{single.name}: {pq.read_metadata(single).num_rows} rows, {single.stat().st_size/1024:.1f} KB")

## 12. Performance notes, and when Spark is the wrong tool

**Caching.** Sections 8–10 recomputed `evs` from Parquet for every action. If a
DataFrame is reused many times, `cache()` keeps it in executor memory after the
first action. It is a hint, not a command, and memory is per executor — on
`local[*]` that is the driver's heap (`spark.driver.memory`, 1 GB by default).

**Adaptive query execution** is on by default in Spark 3+/4: it re-plans shuffles
at run time (coalescing tiny partitions, switching to broadcast joins). The
`AdaptiveSparkPlan` node at the top of every `explain()` is it.

**Honest comparison.** The cell below times the per-minute rollup in Spark and in
DuckDB. Expect DuckDB to win by a wide margin on one machine: no JVM, no
scheduler, no py4j hop, and a vectorised engine designed for exactly this size.
Spark's advantage is that the code in this notebook runs unchanged on 100 machines
and 100 TB. (The row counts differ slightly: DuckDB reads the raw column and
skips the rows section 5 repaired.)

In [ ]:
evs_cached = evs.cache()
t0_ = time.perf_counter(); evs_cached.count(); first = time.perf_counter() - t0_      # materialises the cache
t0_ = time.perf_counter(); evs_cached.count(); second = time.perf_counter() - t0_
print(f"count: {first:.2f} s cold, {second:.2f} s from cache")

t0_ = time.perf_counter()
n_spark = resample(evs_cached, "1 minute").count()
t_spark = time.perf_counter() - t0_

t0_ = time.perf_counter()
n_duck = duckdb.sql(f"""
    SELECT session_id, ts_utc_us // 60000000 AS minute_no, count(*) AS events
    FROM read_parquet('{(ARCHIVE / "events" / "**" / "*.parquet").as_posix()}', hive_partitioning=true)
    WHERE ts_utc_us IS NOT NULL
    GROUP BY 1, 2
""").fetchall()
t_duck = time.perf_counter() - t0_

print(f"per-minute rollup: Spark {n_spark:,} rows in {t_spark:.2f} s | DuckDB {len(n_duck):,} rows in {t_duck:.2f} s")
_ = evs_cached.unpersist()

## 13. Exercises

Each of these is a few lines using only what the notebook already showed.

1. **Absolute window, local time.** Rewrite `slice_abs` to accept naive local
   datetimes and convert them with `zoneinfo.ZoneInfo(LOCAL_TZ)` before `to_us`.
2. **Relative window from the end.** "The last 30 s of every session" — use
   `max("ts_utc_us").over(per_session)`.
3. **Per-game per-minute.** Add `game` to the `groupBy` in `resample` and find the
   minute with the most mouse travel in `cod.exe`.
4. **Bursts, manually.** Reproduce 9a's `bursts` with the `lag` + running-sum
   pattern from 9b (`run_start` when the gap to the previous event exceeds 5 s)
   and check the counts agree with `session_window`.
5. **Marker-anchored windows.** `markers_raw` has `config_changed` events. For
   each marker, compare events-per-second in the 30 s before and after it.
6. **Write, then prune.** Write `runs` from 9b partitioned by `game`, read it back
   with Spark, filter on one game and confirm with `explain()` that only that
   directory is scanned.

## 14. Shut down

`spark.stop()` releases the JVM and the port 4040 UI. Leaving sessions running is
the usual reason a second notebook complains that port 4040 is in use (it will
pick 4041, which is harmless but confusing).

In [ ]:
spark.stop()
print("stopped")